# Chapter 08 — Selection widgets: Combobox, Checkbutton, Radiobutton, Spinbox, and Scale

**Book role:** Core GUI  
**Suggested book space:** about 7 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Match selection widgets to the semantics of the choice.
- Use readonly lists for constrained filters.
- Design defaults that are safe and explicit.
- Convert selections to analysis parameters.

## 1. Choice versus free text

Use `ttk.Combobox` when a user should select one value from a known set; set `state="readonly"` when arbitrary text is not allowed. Use radio buttons for a small mutually exclusive group that benefits from seeing all options. Use checkbuttons for independent Boolean options. A Scale is useful for a numeric range where interactive adjustment matters; a Spinbox is often preferable for discrete integer values.

The widget should embody the meaning of the data, not just look convenient. A region filter may include “All regions,” which is an application sentinel rather than a literal region in the database. A category filter needs a defined meaning for “Uncategorized.” Keep that mapping explicit in the query or service layer.

### Worked example 1: Collect multiple filter options

This pattern creates a plain filter dictionary. The service layer can map “All regions” to no region constraint and use the metric name to select a computation.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Filter controls")
root.geometry("440x250")
region = tk.StringVar(value="All regions")
metric = tk.StringVar(value="Revenue")
show_zero = tk.BooleanVar(value=False)
minimum = tk.IntVar(value=0)

frame = ttk.Frame(root, padding=16)
frame.pack(fill="both", expand=True)
ttk.Label(frame, text="Region").grid(row=0, column=0, sticky="w", pady=5)
ttk.Combobox(frame, textvariable=region, state="readonly", values=("All regions", "North", "South", "East", "West")).grid(row=0, column=1, sticky="ew")
ttk.Label(frame, text="Metric").grid(row=1, column=0, sticky="w", pady=5)
for col, value in enumerate(("Revenue", "Profit", "Units"), start=1):
    ttk.Radiobutton(frame, text=value, value=value, variable=metric).grid(row=1, column=col, sticky="w")
ttk.Checkbutton(frame, text="Include zero-sales rows", variable=show_zero).grid(row=2, column=0, columnspan=3, sticky="w", pady=6)
ttk.Label(frame, text="Minimum revenue").grid(row=3, column=0, sticky="w", pady=5)
ttk.Spinbox(frame, from_=0, to=1000000, increment=500, textvariable=minimum, width=12).grid(row=3, column=1, sticky="w")
frame.columnconfigure(1, weight=1)

def apply() -> None:
    print({"region": region.get(), "metric": metric.get(), "show_zero": show_zero.get(), "minimum": minimum.get()})
ttk.Button(frame, text="Apply", command=apply).grid(row=4, column=1, sticky="e", pady=10)
root.mainloop()

## 2. Defaults, empties, and range semantics

A selected default affects results. Starting with an arbitrary region can make a dashboard look like it reports all data when it does not. Prefer a clearly labeled `All` option where appropriate and display the effective filter state in the interface. For dates and numeric ranges, define whether endpoints are inclusive and whether an empty value means “no constraint” or “invalid input.”

Selection controls should be revalidated in the service layer because values can also arrive from saved configuration or tests. If a Combobox value is used in a SQL query, pass it as a parameter; do not concatenate user-selected text into SQL. UI options are not a reason to skip parameterized queries.

### Worked example 2: Schedule a debounced search

The 300 ms delay is a debounce, not a database worker. Replace `perform_search` with a fast local operation or dispatch slow work to a worker thread.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
query = tk.StringVar()
pending = None
result = tk.StringVar(value="Type to search after a short pause")


def schedule_search(*_args) -> None:
    global pending
    if pending is not None:
        root.after_cancel(pending)
    pending = root.after(300, perform_search)


def perform_search() -> None:
    global pending
    pending = None
    result.set(f"Ready to search for: {query.get().strip()!r}")

query.trace_add("write", schedule_search)
ttk.Entry(root, textvariable=query, width=40).pack(padx=12, pady=12)
ttk.Label(root, textvariable=result, padding=12).pack()
root.mainloop()

## 3. Dependent controls and update policy

Some options depend on one another: selecting a country may narrow the city list; selecting a dataset may populate columns; choosing a chart type may change available encodings. Update dependent controls deliberately and keep the selected value valid after the option list changes. Do not perform a full database request on every tiny interaction if a clear Apply button is more appropriate.

For live search, consider a short debounce: schedule the search with `after`, cancel the previously scheduled callback when the user types again, then query after the input has settled. Debouncing reduces unnecessary work; it does not replace background execution when the operation itself is slow.

## Guided lab: Lab: filter semantics

Create a filter panel for a sales DataFrame using region, category, minimum revenue, and a checkbox for including cancelled transactions. Define defaults and sentinel behavior. The filter service must accept plain Python values and return a DataFrame; add tests for “All regions,” empty category, zero minimum, and cancelled-only data.

## Review questions and practice

1. When should a Combobox be readonly?
2. Why does an `All` option need explicit semantics?
3. What does debouncing reduce, and what does it not solve?
4. Why must selection values still be validated in the service layer?

## Further reading

- Ttk widgets: https://docs.python.org/3/library/tkinter.ttk.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.